# Reti neurali ricorrenti

Nel modulo precedente, abbiamo trattato rappresentazioni semantiche ricche del testo. L'architettura che abbiamo usato cattura il significato aggregato delle parole in una frase, ma non tiene conto dell'**ordine** delle parole, perché l'operazione di aggregazione che segue gli embedding rimuove questa informazione dal testo originale. Poiché questi modelli non sono in grado di rappresentare l'ordinamento delle parole, non possono risolvere compiti più complessi o ambigui come la generazione di testo o il question answering.

Per catturare il significato di una sequenza di testo, useremo un'architettura di rete neurale chiamata **rete neurale ricorrente**, o RNN. Quando si usa una RNN, facciamo passare la frase attraverso la rete un token alla volta, e la rete produce uno **stato**, che poi passiamo di nuovo alla rete insieme al token successivo.

![Immagine che mostra un esempio di generazione con rete neurale ricorrente.](../../../../../translated_images/it/rnn.27f5c29c53d727b5.webp)

Data la sequenza di input di token $X_0,\dots,X_n$, la RNN crea una sequenza di blocchi di rete neurale, e addestra questa sequenza end-to-end usando la retropropagazione. Ogni blocco di rete prende in input una coppia $(X_i,S_i)$ e produce come risultato $S_{i+1}$. Lo stato finale $S_n$ o l'output $Y_n$ viene passato a un classificatore lineare per produrre il risultato. Tutti i blocchi di rete condividono gli stessi pesi, e sono addestrati end-to-end usando una singola passata di retropropagazione.

> La figura sopra mostra la rete neurale ricorrente nella forma "srotolata" (a sinistra), e nella rappresentazione ricorrente più condensata (a destra). È importante capire che tutte le celle RNN hanno gli stessi **pesi condivisibili**.

Poiché i vettori di stato $S_0,\dots,S_n$ sono passati attraverso la rete, la RNN è in grado di apprendere dipendenze sequenziali tra le parole. Per esempio, quando la parola *not* compare da qualche parte nella sequenza, essa può imparare a negare certi elementi all'interno del vettore di stato.

All'interno, ogni cella RNN contiene due matrici di peso: $W_H$ e $W_I$, e un bias $b$. A ogni passo della RNN, data l'input $X_i$ e lo stato di input $S_i$, lo stato di output si calcola come $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, dove $f$ è una funzione di attivazione (spesso $\tanh$).

> Per problemi come la generazione di testo (che tratteremo nell'unità successiva) o la traduzione automatica vogliamo anche ottenere un valore di output a ogni passo della RNN. In questo caso, c’è anche un’altra matrice $W_O$, e l’output si calcola come $Y_i=f(W_O\times S_i+b_O)$.

Vediamo come le reti neurali ricorrenti possano aiutarci a classificare il nostro dataset di notizie.

> Per l'ambiente sandbox, dobbiamo eseguire la cella seguente per assicurarci che la libreria richiesta sia installata e che i dati siano precaricati. Se stai eseguendo localmente, puoi saltare la cella seguente.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Quando si addestrano modelli di grandi dimensioni, l'allocazione della memoria GPU può diventare un problema. Potremmo anche dover sperimentare con diverse dimensioni dei minibatch, in modo che i dati si adattino alla memoria della nostra GPU, ma l'addestramento sia comunque abbastanza veloce. Se stai eseguendo questo codice sulla tua macchina GPU, puoi provare a regolare la dimensione del minibatch per velocizzare l'addestramento.

> **Nota**: Alcune versioni dei driver NVidia sono note per non rilasciare la memoria dopo l'addestramento del modello. Stiamo eseguendo diversi esempi in questo notebook, e ciò potrebbe causare l'esaurimento della memoria in alcune configurazioni, specialmente se stai facendo esperimenti personali all'interno dello stesso notebook. Se incontri errori strani quando inizi ad addestrare il modello, potresti voler riavviare il kernel del notebook.


In [3]:
batch_size = 16
embed_size = 64

## Classificatore RNN semplice

Nel caso di una RNN semplice, ogni unità ricorrente è una rete lineare semplice, che prende in input un vettore di input e un vettore di stato, e produce un nuovo vettore di stato. In Keras, questo può essere rappresentato dal livello `SimpleRNN`.

Sebbene si possano passare token codificati one-hot direttamente al livello RNN, non è una buona idea a causa dell'elevata dimensionalità. Pertanto, useremo un livello di embedding per ridurre la dimensionalità dei vettori di parole, seguito da un livello RNN, e infine un classificatore `Dense`.

> **Nota**: In casi in cui la dimensionalità non è così elevata, ad esempio quando si utilizza la tokenizzazione a livello di carattere, potrebbe avere senso passare direttamente i token codificati one-hot nella cella RNN.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Nota:** Qui usiamo un layer di embedding non addestrato per semplicità, ma per risultati migliori possiamo usare un layer di embedding preaddestrato con Word2Vec, come descritto nell'unità precedente. Sarebbe un buon esercizio per te adattare questo codice per lavorare con embedding preaddestrati.

Ora addestriamo il nostro RNN. Gli RNN in generale sono piuttosto difficili da addestrare, perché una volta che le celle RNN vengono srotolate lungo la lunghezza della sequenza, il numero risultante di layer coinvolti nella backpropagation è piuttosto grande. Quindi dobbiamo selezionare un tasso di apprendimento più basso e addestrare la rete su un dataset più grande per produrre buoni risultati. Questo può richiedere molto tempo, quindi è preferibile usare una GPU.

Per accelerare le cose, addestreremo il modello RNN solo sui titoli delle notizie, omettendo la descrizione. Puoi provare ad addestrare con la descrizione e vedere se riesci a far addestrare il modello.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Nota** che l’accuratezza qui sarà probabilmente minore, poiché stiamo addestrando solo su titoli di notizie.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Ora che stiamo usando la mascheratura, possiamo addestrare il modello sull'intero set di dati di titoli e descrizioni.

> **Nota**: Hai notato che abbiamo utilizzato un vettorizzatore addestrato sui titoli delle notizie, e non sull'intero corpo dell'articolo? Potenzialmente, questo può causare l'ignoramento di alcuni token, quindi è meglio riaddestrare il vettorizzatore. Tuttavia, potrebbe avere solo un effetto molto piccolo, quindi manterremo il vettorizzatore pre-addestrato precedente per semplicità.


## LSTM: Memoria a breve e lungo termine

Uno dei principali problemi delle RNN è il **vanishing gradient**. Le RNN possono essere piuttosto lunghe e possono avere difficoltà a propagare i gradienti fino al primo strato della rete durante la backpropagation. Quando ciò accade, la rete non riesce ad apprendere le relazioni tra token distanti. Un modo per evitare questo problema è introdurre una **gestione esplicita dello stato** utilizzando **porte**. Le due architetture più comuni che introducono porte sono la **memoria a breve e lungo termine** (LSTM) e la **gated relay unit** (GRU). Qui tratteremo le LSTM.

![Immagine che mostra un esempio di cella a memoria a breve e lungo termine](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Una rete LSTM è organizzata in modo simile a una RNN, ma ci sono due stati che vengono passati da strato a strato: lo stato effettivo $c$ e il vettore nascosto $h$. In ogni unità, il vettore nascosto $h_{t-1}$ si combina con l'input $x_t$, e insieme controllano cosa succede allo stato $c_t$ e all'output $h_t$ attraverso le **porte**. Ogni porta ha un'attivazione sigmoidale (con output nel range $[0,1]$), che può essere pensata come una maschera bit a bit quando moltiplicata per il vettore di stato. Le LSTM hanno le seguenti porte (da sinistra a destra nell'immagine qui sopra):
* **porta di dimenticanza** che determina quali componenti del vettore $c_{t-1}$ dobbiamo dimenticare e quali trasmettere.
* **porta di input** che determina quanta informazione dal vettore di input e dal vettore nascosto precedente dovrebbe essere incorporata nello stato.
* **porta di output** che prende il nuovo vettore di stato e decide quali dei suoi componenti saranno usati per produrre il nuovo vettore nascosto $h_t$.

I componenti dello stato $c$ possono essere pensati come flag che possono essere attivati e disattivati. Per esempio, quando incontriamo il nome *Alice* nella sequenza, presumiamo che si riferisca a una donna, e attiviamo il flag nello stato che segnala la presenza di un sostantivo femminile nella frase. Quando incontriamo inoltre le parole *e Tom*, attiveremo il flag che indica la presenza di un sostantivo plurale. Manipolando dunque lo stato possiamo tenere traccia delle proprietà grammaticali della frase.

> **Nota**: Ecco una grande risorsa per comprendere il funzionamento interno delle LSTM: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) di Christopher Olah.

Sebbene la struttura interna di una cella LSTM possa sembrare complessa, Keras nasconde questa implementazione all'interno dello strato `LSTM`, quindi l'unica cosa che dobbiamo fare nell'esempio sopra è sostituire lo strato ricorrente:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Nota** che addestrare LSTM è anche piuttosto lento, e potresti non notare molto aumento di precisione all'inizio dell'addestramento. Potrebbe essere necessario continuare l'addestramento per un po' di tempo per raggiungere una buona precisione.


## RNN bidirezionali e multilivello

Nei nostri esempi finora, le reti ricorrenti operano dall'inizio di una sequenza fino alla fine. Questo ci sembra naturale perché segue la stessa direzione in cui leggiamo o ascoltiamo il parlato. Tuttavia, per scenari che richiedono l'accesso casuale alla sequenza di input, ha più senso eseguire il calcolo ricorrente in entrambe le direzioni. Le RNN che permettono calcoli in entrambe le direzioni sono chiamate RNN **bidirezionali**, e possono essere create racchiudendo lo strato ricorrente con uno speciale strato `Bidirectonal`.

> **Nota**: Lo strato `Bidirectional` fa due copie dello strato al suo interno, e imposta la proprietà `go_backwards` di una di queste copie a `True`, facendolo procedere nella direzione opposta lungo la sequenza.

Le reti ricorrenti, unidirezionali o bidirezionali, catturano schemi all'interno di una sequenza e li memorizzano in vettori di stato o li restituiscono come output. Come con le reti convoluzionali, possiamo costruire un altro strato ricorrente dopo il primo per catturare schemi di livello superiore, costruiti da schemi di livello inferiore estratti dal primo strato. Questo ci porta alla nozione di **RNN multilivello**, che consiste di due o più reti ricorrenti, dove l'output dello strato precedente viene passato come input allo strato successivo.

![Immagine che mostra una RNN multilivello long-short-term-memory](../../../../../translated_images/it/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Immagine tratta da [questo meraviglioso post](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) di Fernando López.*

Keras rende la costruzione di queste reti un compito semplice, perché devi semplicemente aggiungere più strati ricorrenti al modello. Per tutti gli strati tranne l'ultimo, dobbiamo specificare il parametro `return_sequences=True`, perché abbiamo bisogno che lo strato restituisca tutti gli stati intermedi, e non solo lo stato finale del calcolo ricorrente.

Costruiamo una LSTM bidirezionale a due livelli per il nostro problema di classificazione.

> **Nota** questo codice richiede di nuovo molto tempo per completarsi, ma ci dà la massima accuratezza che abbiamo visto finora. Quindi forse vale la pena aspettare e vedere il risultato.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN per altri compiti

Fino ad ora, ci siamo concentrati sull'uso delle RNN per classificare sequenze di testo. Ma possono gestire molti altri compiti, come la generazione di testo e la traduzione automatica &mdash; considereremo questi compiti nella prossima unità.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
